# Construcción de todas las variables nuevas

Hasta acá se crearon tres grupos de variables en notebooks separados: las temporales (por ejemplo, el frío de ayer o el promedio de los últimos siete días), las razones de planta (por ejemplo, cuánto del consumo de la planta corresponde al frío) y las interacciones (productos entre pares de variables importantes). Este notebook las junta en dos funciones que se usan siempre en el mismo orden. La primera, `fitear_reglas_de_variables`, mira solamente el tramo de entrenamiento y aprende lo que hace falta para crear las variables: los umbrales de las razones y cuáles son las diez variables base de las interacciones. La segunda, `transform_variables`, toma esas reglas y agrega las variables nuevas a cualquier tabla. Separarlas asegura que lo aprendido sale solo de entrenamiento y que se aplica igual a validación, prueba y datos nuevos, como pide la consigna.

El recorrido completo, desde los archivos originales, es: leer los Excel y armar la tabla de un día por fila, dividirla en entrenamiento, validación y prueba, limpiarla, y recién entonces usar estas dos funciones. Las funciones de este notebook no leen los Excel ni limpian: parten de la tabla que ya salió de la limpieza.

## 1. Preparación: tabla limpia y división

Esta sección repite el recorrido de los notebooks anteriores hasta llegar a la tabla limpia. Se cargan los datos, se dividen en entrenamiento, validación y prueba por fecha, y se limpian los tres tramos con reglas aprendidas únicamente en entrenamiento. Se obtiene así la tabla de partida de este notebook. La primera tabla es un control: debe dar la misma cantidad de días y de columnas que en el notebook de preprocesamiento (1.212 días, 276 columnas), lo que confirma que se parte del mismo punto.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append("../src")
from auxiliar_functions import tabla_simple
from preprocessing_pipeline import (cargar_tabla_completa, dividir_por_fecha,
                                    fitear_reglas_de_variables, transform_variables)
from limpieza import aprender_reglas_de_limpieza, limpiar
from variables_temporales import crear_variables_temporales
from variables_planta import aprender_reglas_de_planta, crear_variables_planta, NOMBRES_NUEVAS as NOMBRES_PLANTA
from variables_interacciones import aprender_reglas_de_interacciones, crear_interacciones

tabla_completa = cargar_tabla_completa(Path("../data"))
X_train, X_val, X_test = dividir_por_fecha(tabla_completa)
tabla_limpia = limpiar(tabla_completa, aprender_reglas_de_limpieza(X_train)).copy()
train_limpio, val_limpio, test_limpio = dividir_por_fecha(tabla_limpia)

control = pd.DataFrame({"Valor": {
    "Días de la tabla limpia": len(tabla_limpia), "Columnas de la tabla limpia": tabla_limpia.shape[1],
    "Días de entrenamiento": len(train_limpio), "Días de validación": len(val_limpio),
    "Días de prueba": len(test_limpio)}}).rename_axis("Control").reset_index()
print("Tabla 1. Control de la tabla de partida (debe dar 1.212 días y 276 columnas)")
display(tabla_simple(control, precision=0))

Tabla 1. Control de la tabla de partida (debe dar 1.212 días y 276 columnas)


,Control,Valor
0,Días de la tabla limpia,1212
1,Columnas de la tabla limpia,276
2,Días de entrenamiento,848
3,Días de validación,182
4,Días de prueba,182


## 2. Primera función: aprender las reglas

`fitear_reglas_de_variables` recibe solamente el tramo de entrenamiento limpio. Hace dos cosas con él: calcula el umbral de cada denominador de las razones de planta (una razón queda vacía cuando su denominador es muy chico) y elige las diez variables base de las interacciones, las que más se parecen al frío de mañana sin repetirse entre sí. No guarda nada adentro de la función: devuelve un diccionario con las dos reglas, y ese diccionario es lo que después se le entrega a la segunda función. Las tres tablas siguientes muestran qué contiene: primero la forma general del diccionario, después los umbrales y por último las diez variables base.

In [2]:
reglas = fitear_reglas_de_variables(train_limpio)        # solo con entrenamiento

contenido = pd.DataFrame([
    {"Regla": "planta", "Qué guarda": "Umbral de cada denominador de las razones",
     "Cantidad de elementos": len(reglas["planta"]["umbral_denominador"])},
    {"Regla": "interacciones", "Qué guarda": "Variables base de las interacciones",
     "Cantidad de elementos": len(reglas["interacciones"]["variables_base"])}])
print("Tabla 2. Qué devuelve fitear_reglas_de_variables (un diccionario con dos reglas)")
display(tabla_simple(contenido, precision=0))

umbrales = pd.DataFrame({"Umbral": reglas["planta"]["umbral_denominador"]}).rename_axis("Denominador").reset_index()
print("Tabla 3. Regla de planta: umbral de cada denominador (10 % de su mediana de entrenamiento)")
display(tabla_simple(umbrales, precision=0))

base = pd.DataFrame({"Variable base": reglas["interacciones"]["variables_base"]})
print("Tabla 4. Regla de interacciones: las variables base elegidas con entrenamiento")
display(tabla_simple(base, precision=0))

Tabla 2. Qué devuelve fitear_reglas_de_variables (un diccionario con dos reglas)


,Regla,Qué guarda,Cantidad de elementos
0,planta,Umbral de cada denominador de las razones,2
1,interacciones,Variables base de las interacciones,10


Tabla 3. Regla de planta: umbral de cada denominador (10 % de su mediana de entrenamiento)


,Denominador,Umbral
0,Consolidado Produccion | Hl Cerveza Envasada,980
1,Planta (Kw),7006


Tabla 4. Regla de interacciones: las variables base elegidas con entrenamiento


,Variable base
0,Frio (Kw)
1,dia_anio_coseno
2,frio_hace_2_dias
3,Consolidado KPI | ET Bodega/Hl
4,Consolidado Agua | Agua Planta (Hl)
5,Totalizadores Energia | KW Cond 5. 6 y 9
6,Totalizadores Energia | KW Mycom 7
7,Totalizadores Energia | KW Trafo 9
8,Consolidado GasVapor | ET Servicios (Mj)
9,Totalizadores Energia | KW Mycom 2


## 3. Segunda función: aplicar las reglas

`transform_variables` recibe dos cosas: el diccionario de reglas de la sección anterior y la tabla limpia completa, con los tres tramos juntos y en orden cronológico. No aprende nada: crea las variables temporales, después las razones de planta y por último las interacciones.

La tabla tiene que venir completa y en orden por una razón concreta. Un rezago, como el frío de ayer, se calcula tomando el valor de la fila de arriba, y el promedio de siete días toma las seis filas de arriba y la actual. Por eso la primera fila de validación necesita ver, en la misma tabla, la última de entrenamiento. Con cinco días de ejemplo, y el corte después del tercero:

| Fila | Tramo | Frío de hoy | Frío de ayer si se calcula con la tabla completa | Frío de ayer si se calcula solo con validación |
|---|---|---|---|---|
| 1 | entrenamiento | 100 | vacío (no hay fila anterior) | no aplica |
| 2 | entrenamiento | 110 | 100 | no aplica |
| 3 | entrenamiento | 120 | 110 | no aplica |
| 4 | validación | 130 | 120 | vacío |
| 5 | validación | 125 | 130 | 130 |

Si la función recibiera solo la validación, el día 4 no tendría fila anterior y su frío de ayer quedaría vacío aunque el dato existe; con el promedio de siete días pasaría lo mismo en los primeros seis días. Mirar hacia atrás no es una fuga de datos, porque entrenamiento nunca ve filas de validación: solo validación ve filas de entrenamiento, que son el pasado. El orden importa porque el rezago toma la fila de arriba sin mirar la fecha, y por eso la función se detiene con un error si la tabla no está en orden. La tabla de abajo muestra cómo crece la tabla y cuántas columnas aporta cada paso.

In [3]:
tabla_con_variables = transform_variables(tabla_limpia, reglas)
tr, va, te = dividir_por_fecha(tabla_con_variables)

pasos = pd.DataFrame([
    {"Paso": "Tabla limpia (partida)", "Columnas": tabla_limpia.shape[1]},
    {"Paso": "Variables temporales agregadas", "Columnas": 10},
    {"Paso": "Razones de planta agregadas", "Columnas": len(NOMBRES_PLANTA)},
    {"Paso": "Interacciones agregadas", "Columnas": len(reglas["interacciones"]["variables_base"]) * (len(reglas["interacciones"]["variables_base"]) - 1) // 2},
    {"Paso": "Tabla con todas las variables (resultado)", "Columnas": tabla_con_variables.shape[1]}])
print("Tabla 5. Columnas que aporta cada paso de transform_variables")
display(tabla_simple(pasos, precision=0))

Tabla 5. Columnas que aporta cada paso de transform_variables


,Paso,Columnas
0,Tabla limpia (partida),276
1,Variables temporales agregadas,10
2,Razones de planta agregadas,6
3,Interacciones agregadas,45
4,Tabla con todas las variables (resultado),337


## 4. Controles

Esta sección comprueba que las dos funciones hacen lo que se espera, y todos los controles deben dar verdadero. Los primeros tres revisan la forma de la tabla: que tenga las mismas filas que la tabla limpia, que las columnas originales no hayan cambiado y que la cantidad total de columnas sea la esperada. El cuarto compara con el camino que se siguió en los notebooks anteriores, paso por paso, y exige que el resultado sea exactamente el mismo. Los dos siguientes comprueban que no se mira el futuro: se alteran a propósito los datos de validación y prueba y se verifica que ni las reglas ni las filas de entrenamiento cambian. El último verifica que las reglas se pueden guardar en un archivo de texto con formato JSON, algo que hará falta para dejar registro de cómo se procesaron los datos.

In [4]:
# camino paso por paso de los notebooks anteriores
reglas_planta_manual = aprender_reglas_de_planta(train_limpio)
manual = crear_variables_planta(crear_variables_temporales(tabla_limpia), reglas_planta_manual)
reglas_inter_manual = aprender_reglas_de_interacciones(dividir_por_fecha(manual)[0])
manual = crear_interacciones(manual, reglas_inter_manual)

# se altera el futuro (validación y prueba) a propósito
alterada = tabla_limpia.copy()
columnas_alteradas = ["Frio (Kw)", "Planta (Kw)", "Consolidado Produccion | Hl Cerveza Envasada"]
alterada.loc[val_limpio.index[0]:, columnas_alteradas] = alterada.loc[val_limpio.index[0]:, columnas_alteradas] * 50
train_alterado = dividir_por_fecha(alterada)[0]
con_alteracion = transform_variables(alterada, reglas)

controles = {
    "Mismas filas e índice que la tabla limpia": tabla_con_variables.index.equals(tabla_limpia.index),
    "Las columnas originales no cambian": tabla_con_variables[tabla_limpia.columns].equals(tabla_limpia),
    "Columnas finales = 276 + 10 + 6 + 45 = 337": tabla_con_variables.shape[1] == 337 and tabla_con_variables.columns.is_unique,
    "Da lo mismo que el camino paso por paso de los notebooks anteriores": tabla_con_variables.equals(manual),
    "Cambiar validación y prueba no cambia las reglas": fitear_reglas_de_variables(train_alterado) == reglas,
    "Cambiar validación y prueba no cambia las filas de entrenamiento": dividir_por_fecha(con_alteracion)[0].iloc[:-1].equals(tr.iloc[:-1]),
    "Las reglas se pueden guardar como JSON": bool(json.dumps(reglas)),
}
print("Tabla 6. Controles (todos deben dar True)")
display(tabla_simple(pd.DataFrame({"Resultado": controles}).rename_axis("Control").reset_index()))

Tabla 6. Controles (todos deben dar True)


,Control,Resultado
0,Mismas filas e índice que la tabla limpia,True
1,Las columnas originales no cambian,True
2,Columnas finales = 276 + 10 + 6 + 45 = 337,True
3,Da lo mismo que el camino paso por paso de los notebooks anteriores,True
4,Cambiar validación y prueba no cambia las reglas,True
5,Cambiar validación y prueba no cambia las filas de entrenamiento,True
6,Las reglas se pueden guardar como JSON,True


## 5. Conclusión

Con estas dos funciones, a partir de la tabla limpia se obtiene una tabla con todas las variables: se parte de 276 columnas y se llega a 337, porque se agregan 10 temporales, 6 razones y 45 interacciones. La primera función aprende con entrenamiento y devuelve un diccionario de reglas; la segunda recibe ese diccionario junto con la tabla limpia completa y agrega las variables. Los controles confirman que el resultado coincide con el camino paso por paso y que alterar validación y prueba no cambia lo aprendido.

Esta tabla todavía no es el conjunto final para entrenar: conserva los valores vacíos, no está transformada ni escalada, y tiene muchas más columnas de las que conviene usar. Lo que sigue es elegir con cuáles se queda el modelo (selección de variables) y completar los vacíos, transformar y escalar dentro de un `Pipeline` de scikit-learn (ADR 0008 y 0010).